In [31]:
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [32]:
X_train_processed = joblib.load(
    "../data/outputs/X_train_processed.pkl"
)

X_test_processed = joblib.load(
    "../data/outputs/X_test_processed.pkl"
)

y_train = joblib.load(
    "../data/outputs/y_train.pkl"
)

y_test = joblib.load(
    "../data/outputs/y_test.pkl"
)

feature_names = joblib.load(
    "../data/outputs/feature_names.pkl"
)

print("Training data:", X_train_processed.shape)
print("Testing data :", X_test_processed.shape)
print("Features     :", len(feature_names))

Training data: (9600, 77)
Testing data : (2400, 77)
Features     : 77


In [28]:
TARGET = "approved"

y = df[TARGET].copy()
X = df.drop(columns=[TARGET]).copy()

# Create model pipelines

In [33]:
logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

print("3 models created successfully.")

3 models created successfully.


In [34]:
logistic_model.fit(
    X_train_processed,
    y_train
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

gradient_boosting_model.fit(
    X_train_processed,
    y_train
)

print("All models trained successfully.")

All models trained successfully.


In [35]:
logistic_pred = logistic_model.predict(X_test_processed)
logistic_prob = logistic_model.predict_proba(
    X_test_processed
)[:, 1]

rf_pred = random_forest_model.predict(X_test_processed)
rf_prob = random_forest_model.predict_proba(
    X_test_processed
)[:, 1]

gb_pred = gradient_boosting_model.predict(X_test_processed)
gb_prob = gradient_boosting_model.predict_proba(
    X_test_processed
)[:, 1]

print("Predictions generated successfully.")

Predictions generated successfully.


In [36]:
def evaluate_model(
    name,
    y_true,
    predictions,
    probabilities
):
    return {
        "Model": name,
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            probabilities
        ),
        "PR_AUC": average_precision_score(
            y_true,
            probabilities
        )
    }

In [37]:
results = pd.DataFrame([
    evaluate_model(
        "Logistic Regression",
        y_test,
        logistic_pred,
        logistic_prob
    ),
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred,
        rf_prob
    ),
    evaluate_model(
        "Gradient Boosting",
        y_test,
        gb_pred,
        gb_prob
    )
])

results.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Logistic Regression,0.7538,0.6654,0.7626,0.7107,0.8485,0.7921
1,Random Forest,0.7554,0.6845,0.7111,0.6976,0.8342,0.7727
2,Gradient Boosting,0.7671,0.7304,0.6544,0.6903,0.8443,0.7880


In [38]:
print("========== LOGISTIC REGRESSION ==========")
print(confusion_matrix(y_test, logistic_pred))

print("\n========== RANDOM FOREST ==========")
print(confusion_matrix(y_test, rf_pred))

print("\n========== GRADIENT BOOSTING ==========")
print(confusion_matrix(y_test, gb_pred))

========== LOGISTIC REGRESSION ==========
[[1083  365]
 [ 226  726]]

========== RANDOM FOREST ==========
[[1136  312]
 [ 275  677]]

========== GRADIENT BOOSTING ==========
[[1218  230]
 [ 329  623]]


In [39]:
print("========== LOGISTIC REGRESSION ==========")
print(
    classification_report(
        y_test,
        logistic_pred,
        digits=4
    )
)

print("========== RANDOM FOREST ==========")
print(
    classification_report(
        y_test,
        rf_pred,
        digits=4
    )
)

print("========== GRADIENT BOOSTING ==========")
print(
    classification_report(
        y_test,
        gb_pred,
        digits=4
    )
)

========== LOGISTIC REGRESSION ==========
              precision    recall  f1-score   support

           0     0.8273    0.7479    0.7856      1448
           1     0.6654    0.7626    0.7107       952

    accuracy                         0.7538      2400
   macro avg     0.7464    0.7553    0.7482      2400
weighted avg     0.7631    0.7538    0.7559      2400

========== RANDOM FOREST ==========
              precision    recall  f1-score   support

           0     0.8051    0.7845    0.7947      1448
           1     0.6845    0.7111    0.6976       952

    accuracy                         0.7554      2400
   macro avg     0.7448    0.7478    0.7461      2400
weighted avg     0.7573    0.7554    0.7562      2400

========== GRADIENT BOOSTING ==========
              precision    recall  f1-score   support

           0     0.7873    0.8412    0.8134      1448
           1     0.7304    0.6544    0.6903       952

    accuracy                         0.7671      2400
   macro a

In [40]:
results.to_csv(
    "../data/outputs/model_comparison.csv",
    index=False
)

print("Model comparison saved.")

Model comparison saved.


In [41]:
joblib.dump(
    logistic_model,
    "../data/outputs/logistic_model.pkl"
)

joblib.dump(
    random_forest_model,
    "../data/outputs/random_forest_model.pkl"
)

joblib.dump(
    gradient_boosting_model,
    "../data/outputs/gradient_boosting_model.pkl"
)

print("All trained models saved.")

All trained models saved.


In [43]:
import joblib

joblib.dump(
    logistic_model,
    "../data/outputs/final_model.pkl"
)

print("Final model saved successfully.")

Final model saved successfully.


In [44]:
# ============================================================
# RE-SAVE FINAL MODEL WITH CURRENT SCIKIT-LEARN VERSION
# ============================================================

import sklearn
import joblib
from sklearn.linear_model import LogisticRegression

print("Current scikit-learn version:", sklearn.__version__)

# Recreate the exact final model configuration
final_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train using the already prepared training data
final_model.fit(
    X_train_processed,
    y_train
)

# Save final model
joblib.dump(
    final_model,
    "../data/outputs/final_model.pkl"
)

print("Final model re-trained and saved successfully.")

Current scikit-learn version: 1.7.2
Final model re-trained and saved successfully.


In [45]:
# ============================================================
# VERIFY FINAL MODEL
# ============================================================

import joblib

loaded_model = joblib.load(
    "../data/outputs/final_model.pkl"
)

test_prediction = loaded_model.predict(
    X_test_processed
)

test_probability = loaded_model.predict_proba(
    X_test_processed
)[:, 1]

print("Final model loaded successfully.")
print("Test accuracy:", round(
    accuracy_score(y_test, test_prediction), 4
))
print("Test ROC-AUC:", round(
    roc_auc_score(y_test, test_probability), 4
))

Final model loaded successfully.
Test accuracy: 0.7538
Test ROC-AUC: 0.8485


In [46]:
# ============================================================
# FORCE RE-SAVE FINAL MODEL WITH SCIKIT-LEARN 1.8.0
# ============================================================

import sklearn
import joblib
from sklearn.linear_model import LogisticRegression

print("Current scikit-learn:", sklearn.__version__)

final_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

final_model.fit(X_train_processed, y_train)

joblib.dump(
    final_model,
    "../data/outputs/final_model.pkl"
)

print("Saved final_model.pkl successfully.")

Current scikit-learn: 1.7.2
Saved final_model.pkl successfully.


In [47]:
import joblib
import sklearn

print("Current sklearn:", sklearn.__version__)

model = joblib.load("../data/outputs/final_model.pkl")

print("Model loaded successfully.")
print("Model sklearn version check complete.")

Current sklearn: 1.7.2
Model loaded successfully.
Model sklearn version check complete.


In [48]:
import os
print(os.path.abspath("../data/outputs/final_model.pkl"))

c:\Users\Monalika\OneDrive\Desktop\creditwise\data\outputs\final_model.pkl


In [42]:
results.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Logistic Regression,0.7538,0.6654,0.7626,0.7107,0.8485,0.7921
1,Random Forest,0.7554,0.6845,0.7111,0.6976,0.8342,0.7727
2,Gradient Boosting,0.7671,0.7304,0.6544,0.6903,0.8443,0.7880
